# Creating an A2A Sequential Chain Agent

Now that you have two active agents (Policy Agent and Research Agent), you will orchestrate them. In this exercise, you will create a workflow where a user's query is processed by the Research Agent first, and then the Policy Agent. You will use `ClientFactory` to connect to the servers you started in previous exercises.

In [ ]:
import httpx
from IPython.display import Markdown, display
from a2a.client import (
    Client,
    ClientConfig,
    ClientFactory,
    create_text_message_object,
)
from a2a.types import AgentCard, Artifact, Message, Task
from a2a.utils.message import get_message_text


In [ ]:
host = "localhost"
port_sequence_list = [9998, 9999]

In [ ]:
async def call_agent(prompt, host, port):
    async with httpx.AsyncClient(timeout=100.0) as httpx_client:
        # Step 1: Create a client
        client: Client = await ClientFactory.connect(
            f"http://{host}:{port}",
            client_config=ClientConfig(
                httpx_client=httpx_client,
            ),
        )

        # Step 2: Discover the agent by fetching its card
        agent_card = await client.get_card()

        # Step 3: Create the message using a convenient helper function
        message = create_text_message_object(content=prompt)

        display(Markdown(f"**Sending prompt:** `{prompt}` to {agent_card.name}..."))

        # Step 4: Send the message and await the final response.
        responses = client.send_message(message)

        text_content = ""

        # Step 5: Process the responses from the agent
        async for response in responses:
            if isinstance(response, Message):
                # The agent replied directly with a final message
                print(f"Message ID: {response.message_id}")
                text_content = get_message_text(response)
            # response is a ClientEvent
            elif isinstance(response, tuple):
                task: Task = response[0]
                print(f"Task ID: {task.id}")
                if task.artifacts:
                    artifact: Artifact = task.artifacts[0]
                    print(f"Artifact ID: {artifact.artifact_id}")
                    text_content = get_message_text(artifact)

        if text_content:
            return text_content
        else:

            return "**No final text content received or task did not complete successfully.**"

In [ ]:
async def process_query(query):
    responses = []
    for port in port_sequence_list:
        response = await call_agent(query, host, port)
        responses.append(response)

    display(Markdown("-----\n### Final Response\n-----"))
    for response in responses:
        display(Markdown(response))
    

In [ ]:
await process_query( "How can I get mental health therapy?")